In [2]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [3]:
class CTRDataset(Dataset):
    def __init__(self, dense_tensor, sparse_tensor, label_tensor):
        self.dense = dense_tensor
        self.sparse = sparse_tensor
        self.labels = label_tensor

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.dense[index], self.sparse[index], self.labels[index]


In [11]:
def CTRDataloader(df, dense_cols, sparse_cols, target_col, batch_size=1024, random_state=42):
    train_df, val_df = train_test_split(
        df, test_size=0.2, stratify=df[target_col]
    )

    train_df = train_df.copy()
    val_df = val_df.copy()

    scaler = StandardScaler()

    for col in dense_cols:
        median = train_df[col].median()
        train_df[col] = train_df[col].fillna(median)
        val_df[col] = val_df[col].fillna(median)

    train_dense = scaler.fit_transform(train_df[dense_cols]).astype(np.float32)
    val_dense = scaler.transform(val_df[dense_cols]).astype(np.float32)

    sparse_cardinalities = []

    train_sparse_ohe = []
    val_sparse_ohe = []

    for col in sparse_cols:
        train_df[col] = train_df[col].fillna('<UNKNOWN>').astype(str)
        val_df[col] = val_df[col].fillna('<UNKNOWN').astype(str)

        unique_vals = train_df[col].unique().tolist()

        vocab = {val: idx + 1 for idx, val in enumerate(unique_vals)}
        vocab_size = len(vocab) + 1

        sparse_cardinalities.append(vocab_size)

        train_sparse_ohe.append(train_df[col].map(lambda x: vocab.get(x, 0)).values)
        val_sparse_ohe.append(train_df[col].map(lambda x: vocab.get(x, 0)).values)

    train_sparse = np.column_stack(train_sparse_ohe).astype(np.int64)
    val_sparse = np.column_stack(val_sparse_ohe).astype(np.int64)

    train_labels = train_df[target_col].values.astype(np.float32)
    val_labels = val_df[target_col].values.astype(np.float32)

    train_ds = CTRDataset(
        torch.from_numpy(train_dense), torch.from_numpy(val_dense), torch.from_numpy(train_labels)
    )

    val_ds = CTRDataset(
        torch.from_numpy(val_dense), torch.from_numpy(val_sparse), torch.from_numpy(val_labels)
    )

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, drop_last=True
    )

    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle = False
    )

    return train_loader, val_loader, sparse_cardinalities

    
        

In [12]:
df = pd.read_csv("../../datasets/dataset/train.csv")
dense_cols = [f"integer_feature_{i}" for i in range(1, 14)]
sparse_cols = [f"categorical_feature_{i}" for i in range(1, 27)]
train_loader, val_loader, sparse_cardinalities = CTRDataloader(df, dense_cols=dense_cols, sparse_cols=sparse_cols, target_col="label")


In [13]:
import torch.nn as nn

In [14]:
class DLRM(nn.Module):
    def __init__(self, sparse_cardinalities, num_dense_features=13, embed_dim=16, bottom_mlp_dims=[128, 64, 16], top_mlp_dims=[256, 128, 64], dropout=0.3):
        super().__init__()

        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim) for vocab_size in sparse_cardinalities
        ])

        btm_lyrs = []

        in_dim = num_dense_features
        for hidden_dim in bottom_mlp_dims[:-1]:
            btm_lyrs.append(nn.Linear(in_dim, hidden_dim))
            btm_lyrs.append(nn.BatchNorm1d(hidden_dim))
            btm_lyrs.append(nn.ReLU(hidden_dim))
            in_dim = hidden_dim

        btm_lyrs.append(nn.Linear(in_dim, bottom_mlp_dims[:-1]))
        btm_lyrs.append(nn.ReLU())
        self.btm_mlp = nn.Sequential(*btm_lyrs)

        num_features = len(sparse_cardinalities) + 1
        num_interactions = (num_features * (num_features - 1)) // 2

        triu_i, triu_j = torch.triu_indices(num_features, num_features, offset=1)
        self.register_buffer("triu_i", triu_i)
        self.register_buffer("triu_j", triu_i)

        top_input_dim = num_interactions + num_features

        top_lyrs = []
        in_dim = top_input_dim

        for dim in top_mlp_dims:
            top_lyrs.append(nn.Linear(in_dim, dim))
            top_lyrs.append(nn.BatchNorm1d(dim))
            top_lyrs.append(nn.ReLU())
            top_lyrs.append(nn.Dropout(dropout))
            in_dim = dim

        self.top_mlp = nn.Sequential(*top_lyrs)
        self.final_Linear = nn.Linear(top_mlp_dims[-1], 1)

    def forward(self, dense_x, categ_x):
        B = dense_x.size(0)
        v_0 = self.bottom_mlp(dense_x)

        sparse_embeds = [
            self.embeddings[i](categ_x[:, i]) for i in range(categ_x.size(1))
        ]

        T = torch.stack([v_0]+ sparse_embeds, dim = 1)

        Z = torch.bmm(T, T.transpose(1, 2))
        interactions = Z[:, self.triu_i, self.triu_j]
        combined = torch.cat([interactions, v_0], dim=1)

        out = self.top_mlp(combined)
        return self.final_Linear(out).squeeze(-1)



In [15]:

from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, accuracy_score, f1_score, precision_score, recall_score

def footprints(model):
    total_params = sum(x.numel() for x in model.parameters())
    trainable_params = sum(x.numel() for x in model.parameters() if x.requires_grad)
    param_size_mb = sum(x.numel() * x.element_size() for x in model.parameters()) / (1024 ** 2)
    buffer_size_mb = sum(x.numel() * x.element_size() for x in model.buffers()) / (1024 ** 2)
    total_memory_size = param_size_mb + buffer_size_mb
    return total_params, trainable_params, total_memory_size

In [16]:
def expect_calib_err(y_true, y_prob, n_bins=10):
    bins = np.linspace(0., 1., n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1
    ece=0.0
    for bin in range(n_bins):
        mask = bin_ids == bin
        if np.sum(mask) > 0:
            bin_acc = np.mean(y_true[mask])
            bin_conf = np.mean(y_prob[mask])
            ece += (np.sum(mask) / len(y_true)) * np.abs(bin_acc - bin_conf)

    return ece

In [18]:
from torch.nn import BCEWithLogitsLoss

def training(model, train_loader, val_loader, 
                epochs=5, lr=0.001, weight_decay=1e-4, 
                pos_weight_val=20.0, device="cpu"):
    device="cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    pos_weight = torch.tensor([24], device = device)
    criterion = BCEWithLogitsLoss(pos_weight = pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr = lr, weight_decay=weight_decay)

    total_params, trainable_params, total_mem_size = footprints(model)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.
        sample_processed = 0

        for dense_x, categ_x, labels in train_loader:
            dense_x, categ_x, labels = dense_x.to(device), categ_x.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(dense_x, categ_x)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += (loss.item() * labels.size(0))
            sample_processed += labels.size(0)

        print(f"Epoch: {epoch} | train_loss: {running_loss}")

    model.eval()
    all_probs=[]
    all_targets=[]

    for dense_x, sparse_x, labels in val_loader:
        dense_x, sparse_x = dense_x.to(device), sparse_x.to(device)
        logits = model(dense_x, sparse_x)
        probs = torch.sigmoid(logits)
        all_probs.extend(probs.cpu().numpy())
        all_targets.extend(labels.numpy())

    all_probs = np.array(all_probs)
    all_targets = np.array(all_targets).astype(int)

    roc_auc = roc_auc_score(all_targets, all_probs)
    pr_auc = average_precision_score(all_targets, all_probs)
    logloss = log_loss(all_targets, all_probs)
    ece = expect_calib_err(all_targets, all_probs)

    best_f1, best_t, best_acc, best_rec, best_prec = 0.0, 0.5, 0.0, 0.0, 0.0
    for t in np.arange(0.10, 0.90, 0.01):
        preds = (all_probs >= t).astype(int)
        score = f1_score(all_targets, preds, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_t = t
            best_acc = accuracy_score(all_targets, preds)
            best_rec = recall_score(all_targets, preds, zero_division=0)
            best_prec = precision_score(all_targets, preds, zero_division=0)

    return {
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Log Loss": logloss,
        "ECE": ece,
        "Optimal Threshold": best_t,
        "F1": best_f1,
        "Precision": best_prec,
        "Recall": best_rec,
        "Accuracy": best_acc,
        "Parameters": trainable_params,
        "Memory (MB)": total_mem_size
    }

In [20]:
class DLRM_Ablation(nn.Module):
    def __init__(self, sparse_cardinalities, num_dense_features=13, embed_dim=16, bottom_dim=[128, 64, 16], top_dim=[256, 128, 64], dropout=0.3):
        super().__init__()

        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=v, embedding_dim=embed_dim) for v in sparse_cardinalities
        ])

        bottom_layers=[]
        in_dim = num_dense_features

        for dim in bottom_dim[:-1]:
            bottom_layers.append(nn.Linear(in_dim, dim))
            bottom_layers.append(nn.ReLU())
            in_dim = dim
        bottom_layers.append(nn.Linear(in_dim, bottom_dim[:-1]))
        bottom_layers.append(nn.ReLU())
        self.bottom_mlp = nn.Sequential(*bottom_layers)

        flat_dim = (len(sparse_cardinalities) + 1) * embed_dim

        top_layers = []
        in_dim = flat_dim
        for dim in top_dim:
            top_layers.append(nn.Linear(in_dim, dim))
            top_layers.append(nn.BatchNorm1d(dim))
            top_layers.append(nn.ReLU())
            top_layers.append(nn.Dropout(dropout))
            in_dim = dim
        self.top_mlp = nn.Sequential(*top_layers)
        self.final_Linear = nn.Linear(in_dim, 1)
    
    def forward(self, dense_x, sparse_x):
        v_0 = self.bottom_mlp(dense_x)
        sparse_embeds = [
            nn.embeddings[i](sparse_x[:, i]) for i in range(sparse_x.size(1))
        ]

        flat_vec = torch.cat([v_0] + sparse_embeds, dim=1)
        out = self.top_mlp(flat_vec)
        return self.final_Linear(out).squeeze(-1)
